# OatEasy — Notebook 04: Engine 3 — Recommendation & Optimisation (rule-based)

This notebook is **deterministic, not ML**, as the design recommends. It turns the model outputs into an operational plan:

1. **Demand → prep:** Model 1 median forecast plus a safety buffer gives prep quantities per dish.
2. **Demand → requirement:** forecast orders × recipe quantity ÷ (1 − prep yield loss) gives gross ingredient need.
3. **Net purchase:** need + buffer − usable stock − incoming deliveries, rounded up to the supplier's minimum order quantity.
4. **Ranked actions:** each intervention is scored by avoidable cost × feasibility.
5. **Validation:** the stock balance is re-checked, and the recipe-based requirement is compared with actual usage.

Inputs: `forecast_next_day.csv` (Model 1), `ingredient_risk_next_day.csv` (Model 2), Dataset 2 and Dataset 3 reference tables.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("../data"); OUT = Path("../outputs")

CONFIG = {
    "safety_buffer_pct": 0.10,           # extra headroom on forecast demand
    "usable_stock_rule": "closing stock only if min days-to-expiry >= 1, else 0",
    "feasibility": {"use_existing_stock": 0.9, "hold_purchase": 1.0, "stage_preparation": 0.7},
}
forecast = pd.read_csv(OUT / "forecast_next_day.csv")
risk = pd.read_csv(OUT / "ingredient_risk_next_day.csv")
recipes = pd.read_csv(DATA / "dataset3_recipes.csv")
ing = pd.read_csv(DATA / "ingredient_master.csv")
orders = pd.read_csv(DATA / "dataset3_purchase_orders.csv", parse_dates=["order_date"])
inv = pd.read_csv(DATA / "dataset2_inventory_waste.csv", parse_dates=["date"])
as_of = pd.Timestamp(risk["as_of_date"].iloc[0])
forecast_date = pd.Timestamp(forecast["date"].iloc[0])
print("as of:", as_of.date(), "| planning for:", forecast_date.date())

as of: 2025-12-31 | planning for: 2026-01-01


## 1. Prep plan per dish

In [2]:
fc = forecast.copy()
fc["suggested_prep_qty"] = np.ceil(fc["predicted_orders_median"] * (1 + CONFIG["safety_buffer_pct"])).astype(int)
fc["prep_stage_batch_qty"] = np.ceil(fc["lower_80"]).astype(int)   # cook this first, then replenish in batches
prep = fc[["restaurant_id", "dish_id", "dish_name", "predicted_orders_median", "lower_80", "upper_80",
           "forecast_confidence", "suggested_prep_qty", "prep_stage_batch_qty"]]
prep.sort_values(["restaurant_id", "predicted_orders_median"], ascending=[True, False])

,restaurant_id,dish_id,dish_name,predicted_orders_median,lower_80,upper_80,forecast_confidence,suggested_prep_qty,prep_stage_batch_qty
3,R01,D04,Butter Naan,69.6,48.0,79.0,High,77,48
2,R01,D03,Veg Biryani,57.6,39.0,66.0,High,64,39
0,R01,D01,Paneer Tikka,43.5,29.0,49.0,High,48,29
1,R01,D02,Dal Makhani,41.7,29.0,49.0,High,46,29
5,R01,D06,Palak Paneer,37.8,26.0,46.0,High,42,26
6,R01,D07,Jeera Rice,36.7,23.0,44.0,High,41,23
4,R01,D05,Chole Bhature,35.7,26.0,42.0,High,40,26
7,R01,D08,Masala Dosa,32.6,24.0,42.0,High,36,24
8,R01,D09,Veg Thali,31.3,20.0,36.0,High,35,20
10,R01,D11,Gobi Manchurian,22.9,16.0,27.0,High,26,16


## 2. Ingredient requirement and net purchase

In [3]:
rec = recipes.copy()
rec["effective_qty_per_portion"] = rec["qty_per_portion"] / (1 - rec["prep_yield_loss_pct"])
m = fc.merge(rec, on="dish_id")
m["gross_req_qty"] = m["predicted_orders_median"] * m["effective_qty_per_portion"]
req = (m.groupby(["restaurant_id", "ingredient_id"])["gross_req_qty"].sum().reset_index())
req["safety_buffer_qty"] = req["gross_req_qty"] * CONFIG["safety_buffer_pct"]
req["required_qty"] = req["gross_req_qty"] + req["safety_buffer_qty"]

last = inv[inv["date"] == as_of][["restaurant_id", "ingredient_id", "closing_stock_qty", "min_days_to_expiry"]]
req = req.merge(last, on=["restaurant_id", "ingredient_id"], how="left")
req["usable_stock_qty"] = np.where(req["min_days_to_expiry"].fillna(0) >= 1, req["closing_stock_qty"], 0.0)

pending = orders[orders["actual_delivery_date"].isna()]
incoming = pending.groupby(["restaurant_id", "ingredient_id"])["qty_ordered"].sum().rename("incoming_qty").reset_index()
req = req.merge(incoming, on=["restaurant_id", "ingredient_id"], how="left").fillna({"incoming_qty": 0.0})

req = req.merge(ing[["ingredient_id", "ingredient_name", "unit", "unit_cost_inr", "min_order_qty"]], on="ingredient_id")
req["net_shortfall_qty"] = (req["required_qty"] - req["usable_stock_qty"] - req["incoming_qty"]).clip(lower=0)
req["recommended_purchase_qty"] = np.where(
    req["net_shortfall_qty"] > 0,
    np.ceil(req["net_shortfall_qty"] / req["min_order_qty"]) * req["min_order_qty"], 0.0)
req["recommended_purchase_qty"] = req["recommended_purchase_qty"].round(2)
req["purchase_value_inr"] = (req["recommended_purchase_qty"] * req["unit_cost_inr"]).round(2)
req["purchase_adjustment_qty"] = (req["usable_stock_qty"] + req["incoming_qty"]).clip(upper=req["gross_req_qty"]).round(3)

purchase = req[["restaurant_id", "ingredient_name", "unit", "required_qty", "usable_stock_qty", "incoming_qty",
                "recommended_purchase_qty", "purchase_value_inr", "purchase_adjustment_qty"]].round(3)
purchase.sort_values("purchase_value_inr", ascending=False).head(15)

,restaurant_id,ingredient_name,unit,required_qty,usable_stock_qty,incoming_qty,recommended_purchase_qty,purchase_value_inr,purchase_adjustment_qty
9,R01,Spinach,kg,6.582,0.000,0.0,9.0,540.0,0.000
0,R01,Paneer,kg,10.322,15.240,20.0,0.0,0.0,9.383
1,R01,Tomato,kg,3.602,11.417,0.0,0.0,0.0,3.275
2,R01,Onion,kg,8.497,34.305,0.0,0.0,0.0,7.725
4,R01,Butter,kg,2.536,11.332,0.0,0.0,0.0,2.306
3,R01,Fresh Cream,kg,0.940,3.438,0.0,0.0,0.0,0.854
5,R01,Wheat Flour (Atta),kg,14.636,101.723,0.0,0.0,0.0,13.306
6,R01,Basmati Rice,kg,21.107,47.131,275.0,0.0,0.0,19.188
7,R01,Urad Dal,kg,5.128,9.608,75.0,0.0,0.0,4.662
8,R01,Chickpeas (Kabuli),kg,4.936,7.007,75.0,0.0,0.0,4.487


## 3. Join with ingredient risk and rank actions

In [4]:
# dish-level cost per portion, used to value over-preparation
portion_cost = (rec.merge(ing[["ingredient_id", "unit_cost_inr"]], on="ingredient_id")
                .assign(c=lambda x: x["effective_qty_per_portion"] * x["unit_cost_inr"])
                .groupby("dish_id")["c"].sum().rename("cost_per_portion_inr").reset_index())
prep = prep.merge(portion_cost, on="dish_id")

actions = []
rk = risk.merge(req[["restaurant_id", "ingredient_id", "usable_stock_qty", "recommended_purchase_qty"]],
                on=["restaurant_id", "ingredient_id"], how="left")

# A) use existing stock before it expires
for _, r in rk[(rk["risk_level"].isin(["High", "Medium"])) & (rk["closing_stock_qty"] > 0)].iterrows():
    actions.append({"restaurant_id": r["restaurant_id"], "action": f"Use existing {r['ingredient_name']} first",
                    "type": "use_existing_stock", "item": r["ingredient_name"],
                    "avoidable_cost_inr": r["financial_exposure_inr"],
                    "feasibility": CONFIG["feasibility"]["use_existing_stock"],
                    "why": f"{r['risk_level']} risk ({r['primary_driver']}); "
                           f"min days to expiry {r['min_days_to_expiry']:.0f}"})

# B) hold purchases where stock already covers the requirement
for _, r in rk[(rk["recommended_purchase_qty"] == 0) & (rk["surplus_qty_next_3d"] > 0)].iterrows():
    actions.append({"restaurant_id": r["restaurant_id"], "action": f"Hold purchase of {r['ingredient_name']}",
                    "type": "hold_purchase", "item": r["ingredient_name"],
                    "avoidable_cost_inr": round(r["surplus_qty_next_3d"] * r["unit_cost_inr"] * r["risk_prob_next_3d"], 2),
                    "feasibility": CONFIG["feasibility"]["hold_purchase"],
                    "why": "Usable stock and incoming deliveries already cover tomorrow's requirement"})

# C) stage preparation: only cook the lower-bound portions up front
for _, r in prep.iterrows():
    extra = max(0, r["suggested_prep_qty"] - r["prep_stage_batch_qty"])
    if extra > 0:
        actions.append({"restaurant_id": r["restaurant_id"],
                        "action": f"Stage preparation of {r['dish_name']}",
                        "type": "stage_preparation", "item": r["dish_name"],
                        "avoidable_cost_inr": round(extra * r["cost_per_portion_inr"] * 0.5, 2),
                        "feasibility": CONFIG["feasibility"]["stage_preparation"],
                        "why": f"Cook {r['prep_stage_batch_qty']} now, replenish the other {extra} "
                               f"in batches (forecast {r['lower_80']:.0f}-{r['upper_80']:.0f})"})

actions = pd.DataFrame(actions)
actions["priority_score"] = (actions["avoidable_cost_inr"] * actions["feasibility"]).round(2)
actions = actions.sort_values("priority_score", ascending=False).reset_index(drop=True)
actions.index = actions.index + 1
actions[["restaurant_id", "action", "avoidable_cost_inr", "feasibility", "priority_score", "why"]].head(15)

,restaurant_id,action,avoidable_cost_inr,feasibility,priority_score,why
1,R02,Use existing Butter first,5155.28,0.9,4639.75,High risk (Usage pattern); min days to expiry 34
2,R02,Use existing Paneer first,4932.02,0.9,4438.82,High risk (Usage pattern); min days to expiry 4
3,R01,Use existing Paneer first,4448.28,0.9,4003.45,High risk (Usage pattern); min days to expiry 4
4,R02,Hold purchase of Butter,3568.93,1.0,3568.93,Usable stock and incoming deliveries already c...
5,R01,Use existing Butter first,3697.72,0.9,3327.95,High risk (Usage pattern); min days to expiry 49
6,R01,Hold purchase of Butter,1365.89,1.0,1365.89,Usable stock and incoming deliveries already c...
7,R01,Use existing Spinach first,799.02,0.9,719.12,High risk (Expiry); min days to expiry 0
8,R01,Use existing Yogurt (Curd) first,757.89,0.9,682.10,High risk (Expiry); min days to expiry 1
9,R02,Use existing Yogurt (Curd) first,681.57,0.9,613.41,High risk (Usage pattern); min days to expiry 6
10,R01,Use existing Fresh Cream first,676.20,0.9,608.58,Medium risk (Usage pattern); min days to expiry 4


## 4. Operational report summary

In [5]:
report = {
    "as_of": as_of.strftime("%Y-%m-%d"),
    "planning_date": forecast_date.strftime("%Y-%m-%d"),
    "predicted_orders_total": int(round(fc["predicted_orders_median"].sum())),
    "ingredients_at_risk": int((risk["risk_level"].isin(["High", "Medium"])).sum()),
    "purchase_adjustment_qty": round(float(req["purchase_adjustment_qty"].sum()), 2),
    "recommended_purchase_value_inr": round(float(req["purchase_value_inr"].sum()), 2),
    "potential_waste_avoided_inr_estimated": round(float(actions.loc[actions["type"] != "stage_preparation", "priority_score"].sum()), 2),
    "note": "Estimated, not verified. Based on synthetic data; a live pilot is needed to confirm savings.",
}
json.dump(report, open(OUT / "report_summary.json", "w"), indent=2)
report

{'as_of': '2025-12-31',
 'planning_date': '2026-01-01',
 'predicted_orders_total': 821,
 'ingredients_at_risk': 18,
 'purchase_adjustment_qty': 180.22,
 'recommended_purchase_value_inr': 540.0,
 'potential_waste_avoided_inr_estimated': 27896.0,
 'note': 'Estimated, not verified. Based on synthetic data; a live pilot is needed to confirm savings.'}

## 5. Validation of the engine

In [6]:
# (a) stock balance identity across the whole history
resid = (inv["opening_stock_qty"] + inv["received_qty"] - inv["expired_discard_qty"]
         - inv["spoilage_discard_qty"] - inv["actual_usage_qty"] - inv["overprep_discard_qty"]
         - inv["closing_stock_qty"]).abs().max()
print("(a) stock balance max residual:", round(resid, 4), "-> OK" if resid < 0.01 else "-> FAIL")

# (b) recipe-based requirement vs actual requirement, last 90 days
req_hist = pd.read_csv(DATA / "dataset3_daily_requirements.csv", parse_dates=["date"])
recent = req_hist[req_hist["date"] > req_hist["date"].max() - pd.Timedelta(days=90)]
agg = recent.groupby("ingredient_id")[["planned_requirement_with_yield_loss_qty", "actual_required_qty",
                                       "theoretical_requirement_qty"]].sum()
mape_plan = (agg["planned_requirement_with_yield_loss_qty"] - agg["actual_required_qty"]).abs() / agg["actual_required_qty"]
mape_theo = (agg["theoretical_requirement_qty"] - agg["actual_required_qty"]).abs() / agg["actual_required_qty"]
print(f"(b) requirement error, last 90 days (per-ingredient, aggregate):")
print(f"    ignoring prep yield loss : median {mape_theo.median():.1%}")
print(f"    with yield loss (engine) : median {mape_plan.median():.1%}")

# (c) shortage check: how often did stock run out on the historical data?
print(f"(c) share of restaurant-ingredient-days with a shortage: {(inv['shortage_qty'] > 0).mean():.2%}")

(a) stock balance max residual: 0.001 -> OK
(b) requirement error, last 90 days (per-ingredient, aggregate):
    ignoring prep yield loss : median 5.8%
    with yield loss (engine) : median 0.1%
(c) share of restaurant-ingredient-days with a shortage: 1.55%


## 6. Save the operational plan

In [7]:
prep.to_csv(OUT / "prep_plan.csv", index=False)
purchase.to_csv(OUT / "purchase_plan.csv", index=False)
actions.to_csv(OUT / "operational_plan_actions.csv", index=False)
print("saved prep_plan.csv, purchase_plan.csv, operational_plan_actions.csv, report_summary.json")

saved prep_plan.csv, purchase_plan.csv, operational_plan_actions.csv, report_summary.json
